# 銀河のスペクトルから赤方偏移を自分で測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/cosmology/measure_redshift.ipynb)

ここまでのノートブックでは、カタログに載っている赤方偏移 $z$ をそのまま使ってきました。
このノートブックでは、**SDSS(スローン・デジタル・スカイ・サーベイ)の銀河のスペクトル** から、輝線・吸収線の位置を測って $z$ を自分で求め、SDSS のパイプラインの値と比べます。

$$
1 + z = \frac{\lambda_{\mathrm{obs}}}{\lambda_{\mathrm{rest}}}
$$

このノートブックでやること:

1. 静止波長が分かっている輝線・吸収線のリストをつくる
2. SDSS から、いろいろな赤方偏移の銀河のスペクトルを取ってくる
3. 線のリストを $z$ だけずらした「型紙」をスペクトルと重ね合わせ(相互相関)、$z$ のおおよその値を探す
4. 1 本 1 本の線にガウス関数を当てはめて、$z$ を精密に決める
5. どの線も同じ **倍率** $1+z$ で伸びていること(足し算のずれではないこと)を確かめる
6. SDSS の値と比べる

## 使うデータ

- **SDSS DR17** のスペクトル(SDSS-I/II の分光器のもの)。SDSS のデータは誰でも自由に使えます(使うときは SDSS への謝辞と、Abdurro'uf et al. 2022 を引用)
  https://www.sdss4.org/dr17/

> 注: このノートブックは SDSS のサーバー(skyserver.sdss.org, data.sdss.org)からデータを取ってきます。Colab からは問題なくつながります。

## 0. 準備

In [ ]:
# Colab には astroquery が入っていないのでインストールする
!pip install -q astroquery

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.utils.data import download_file
from scipy.ndimage import median_filter
from scipy.optimize import curve_fit

from astroquery.sdss import SDSS

C_KMS = 299_792.458  # 光速 [km/s]
DATA_RELEASE = 17

## 1. 線のリスト

SDSS のスペクトルの波長は **真空中の波長** なので、線の静止波長も真空中の値を使います(空気中の値より 1〜2 Å ほど長い)。

- **輝線**: 若い星に温められたガス(電離ガス)が出す線。星をつくっている銀河で強い
  - 水素のバルマー線(Hα, Hβ)、酸素・窒素・硫黄の禁制線([O II], [O III], [N II], [S II])
- **吸収線**: 星の大気が吸収する線。星をつくり終えた楕円銀河では、これが主な手がかり
  - カルシウムの H, K 線、G バンド(CH 分子)、マグネシウムの Mg b、ナトリウムの D 線

2 つ目の数字は、相互相関の型紙に使うおおよその強さ(典型的な銀河での相対的な強さ)です。

In [ ]:
EMISSION = {  # 名前: (真空中の静止波長 [Å], 型紙での相対的な強さ)
    "[OII]3728": (3728.48, 1.0),  # 3727.1 と 3729.9 の 2 本が重なったもの
    "Hβ": (4862.68, 1.0),
    "[OIII]4960": (4960.30, 0.5),
    "[OIII]5008": (5008.24, 1.0),
    "[NII]6550": (6549.86, 0.3),
    "Hα": (6564.61, 3.0),
    "[NII]6585": (6585.27, 1.0),
    "[SII]6718": (6718.29, 0.5),
    "[SII]6733": (6732.67, 0.5),
}
ABSORPTION = {
    "Ca II K": (3934.78, 1.0),
    "Ca II H": (3969.59, 1.0),
    "G band": (4305.6, 0.5),
    "Mg b": (5176.7, 0.7),
    "Na D": (5895.6, 0.7),  # 5891.6 と 5897.6 の 2 本が重なったもの
}

## 2. SDSS から銀河のスペクトルを取ってくる

SDSS のデータベース(SkyServer)に SQL で問い合わせて、赤方偏移の範囲ごとに数個ずつ銀河を選びます。

- `class = 'GALAXY'`: パイプラインが銀河と判定したもの
- `zWarning = 0`: パイプラインの赤方偏移に問題がないもの
- `run2d = '26'`: SDSS-I/II の分光器で取ったもの(ファイルの置き場所がそろっているので)

取ってきたあとは、パイプラインの $z$ は **答え合わせのときまで見ない** ことにします。

In [ ]:
Z_BINS = [(0.01, 0.05), (0.05, 0.1), (0.1, 0.15), (0.15, 0.25), (0.25, 0.4), (0.4, 0.6)]
N_PER_BIN = 4

frames = []
for lo, hi in Z_BINS:
    query = f"""
    SELECT TOP {N_PER_BIN} s.plate, s.mjd, s.fiberID, s.ra, s.dec, s.z, s.zErr, s.subClass
    FROM SpecObj AS s
    WHERE s.class = 'GALAXY' AND s.zWarning = 0 AND s.run2d = '26'
      AND s.z BETWEEN {lo} AND {hi}
    """
    frames.append(SDSS.query_sql(query, data_release=DATA_RELEASE).to_pandas())
targets = pd.concat(frames, ignore_index=True)
targets.columns = targets.columns.str.lower()
print(f"{len(targets)} 個の銀河")
targets[["plate", "mjd", "fiberid", "ra", "dec"]]

スペクトルのファイル(`spec-PLATE-MJD-FIBER.fits`)を取ってきます。中身は

- 1 番目の表(`COADD`): `loglam`(波長の常用対数, Å)、`flux`(フラックス, $10^{-17}$ erg/s/cm²/Å)、`ivar`(フラックスの誤差の 2 乗の逆数。0 なら使えない点)
- 2 番目の表(`SPECOBJ`): パイプラインの結果(`Z` など)

です。astroquery でうまく取れないときのために、ファイルの URL を直接組み立てる方法も用意しておきます。

In [ ]:
def download_spectrum(plate, mjd, fiber):
    # SDSS のスペクトル(HDUList)を返す
    try:
        hdus = SDSS.get_spectra(plate=int(plate), mjd=int(mjd), fiberID=int(fiber), data_release=DATA_RELEASE)
        if hdus:
            return hdus[0]
    except Exception as err:  # astroquery の問い合わせが失敗したら、URL を直接組み立てる
        print("astroquery で取れなかったので、URL から直接取ります:", err)
    url = (f"https://data.sdss.org/sas/dr{DATA_RELEASE}/sdss/spectro/redux/26/spectra/"
           f"{int(plate):04d}/spec-{int(plate):04d}-{int(mjd)}-{int(fiber):04d}.fits")
    return fits.open(download_file(url, cache=True))


def read_spectrum(hdul):
    t = hdul[1].data
    lam = 10 ** t["loglam"].astype(float)
    flux = t["flux"].astype(float)
    ivar = t["ivar"].astype(float)
    return lam, flux, ivar


spectra = []
for _, row in targets.iterrows():
    hdul = download_spectrum(row["plate"], row["mjd"], row["fiberid"])
    spectra.append(read_spectrum(hdul))
print(f"{len(spectra)} 本のスペクトルを読み込みました")

いくつか見てみます。連続光(星の光のなめらかな部分)の上に、輝線(上向きのとがり)や吸収線(下向きのくぼみ)が乗っています。
連続光は、幅の広いメディアンフィルタで近似しておきます(細い線は消えて、なめらかな部分だけが残る)。

In [ ]:
def continuum(flux, width=151):
    # 幅 width ピクセル(約 1.5 万 km/s)のメディアンフィルタで連続光を近似する
    return median_filter(flux, size=width, mode="nearest")


fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, i in zip(axes, np.linspace(0, len(spectra) - 1, 3).astype(int)):
    lam, flux, ivar = spectra[i]
    ax.plot(lam, flux, lw=0.5, c="k", label=f"spectrum #{i}")
    ax.plot(lam, continuum(flux), c="tab:orange", label="continuum (median filter)")
    ax.set_ylim(np.percentile(flux, 0.5), np.percentile(flux, 99.8) * 1.3)
    ax.set_ylabel("flux [1e-17 erg/s/cm²/Å]")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("observed wavelength [Å] (vacuum)")
plt.tight_layout()
plt.show()

## 3. 相互相関で $z$ のおおよその値を探す

線のリストを $z$ だけ伸ばした位置にガウス関数を並べた「型紙」をつくり、スペクトル(連続光を引いたもの)とどれだけ重なるかを、$z$ を少しずつ変えながら計算します。
型紙の線がスペクトルの線とそろったところで、重なりが最大になります。

重なりの大きさは、誤差で重みをつけた内積を、型紙の大きさで割ったもの(=信号雑音比のようなもの)にします。

$$
S(z) = \frac{\sum_i w_i\, f_i\, T_i(z)}{\sqrt{\sum_i w_i\, T_i(z)^2}}, \qquad w_i = \text{ivar}_i
$$

吸収線の場合は、スペクトルを連続光で割って 1 を引き、符号を反対にして(くぼみを山にして)同じことをします。

In [ ]:
Z_GRID = np.arange(0.0, 0.7, 0.0002)


def xcorr_score(lam, signal, weight, lines, z_grid=Z_GRID, sigma_kms=150):
    # 各 z について S(z) を返す
    ln_lam = np.log(lam)
    sig_ln = sigma_kms / C_KMS  # ln λ でのガウス関数の幅
    sw = signal * weight
    score = np.zeros(len(z_grid))
    for k, z in enumerate(z_grid):
        tmpl = np.zeros_like(lam)
        for lam0, amp in lines.values():
            mu = np.log(lam0 * (1 + z))
            if ln_lam[0] < mu < ln_lam[-1]:
                tmpl += amp * np.exp(-0.5 * ((ln_lam - mu) / sig_ln) ** 2)
        norm = np.sqrt(np.sum(weight * tmpl ** 2))
        score[k] = np.sum(sw * tmpl) / norm if norm > 0 else 0.0
    return score


def find_z(lam, flux, ivar):
    # 輝線と吸収線の両方で相互相関をとり、(輝線の S(z), 吸収線の S(z)) を返す
    cont = continuum(flux)
    good = (ivar > 0) & (cont > 0)
    w = np.where(good, ivar, 0.0)
    s_em = xcorr_score(lam, flux - cont, w, EMISSION)
    # 連続光で割ったスペクトルの誤差は 1/(ivar cont²) になる
    s_abs = xcorr_score(lam, -(flux / np.where(good, cont, 1) - 1), w * cont ** 2, ABSORPTION, sigma_kms=300)
    return s_em, s_abs


def peak_significance(score):
    # ピークの高さを、ピーク付近を除いた S(z) のばらつき(中央値からの差の中央値)で割ったもの
    k = np.argmax(score)
    mask = np.abs(np.arange(len(score)) - k) > 50
    mad = np.median(np.abs(score[mask] - np.median(score[mask]))) * 1.4826
    return (score[k] - np.median(score[mask])) / mad


EMISSION_THRESHOLD = 8.0  # これ以上はっきりしたピークなら「輝線がある」とみなす

# 全部のスペクトルで計算しておく。例には、輝線のピークがはっきりしているもののうち、
# (相互相関で見つけた)z が一番大きいものを使う(5 章で「掛け算」のずれが見やすいように)
scores = [find_z(*s) for s in spectra]
sig_em_all = np.array([peak_significance(s_em) for s_em, _ in scores])
z_em_all = np.array([Z_GRID[np.argmax(s_em)] for s_em, _ in scores])
has_em = sig_em_all >= EMISSION_THRESHOLD
i_show = int(np.argmax(np.where(has_em, z_em_all, -1))) if has_em.any() else int(np.argmax(sig_em_all))
lam, flux, ivar = spectra[i_show]
s_em, s_abs = scores[i_show]
print(f"例に使うスペクトル: #{i_show}")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(Z_GRID, s_em)
axes[0].set_ylabel("S(z): emission lines")
axes[1].plot(Z_GRID, s_abs, c="tab:green")
axes[1].set_ylabel("S(z): absorption lines")
axes[1].set_xlabel("trial redshift z")
plt.show()
print(f"輝線の相互相関の最大: z = {Z_GRID[np.argmax(s_em)]:.4f} (S = {s_em.max():.1f})")
print(f"吸収線の相互相関の最大: z = {Z_GRID[np.argmax(s_abs)]:.4f} (S = {s_abs.max():.1f})")

鋭いピークが 1 つ立てば、そこがおおよその $z$ です。
輝線の弱い銀河では、輝線の $S(z)$ はぎざぎざのままで、吸収線の $S(z)$ のほうにはっきりしたピークが出ます。
ピークの「目立ち具合」は、ピークの高さを、ほかの $z$ でのばらつき(雑音の大きさ)で割ったもので測ります(上の `peak_significance`)。

In [ ]:
print(f"輝線のピークの目立ち具合: {peak_significance(s_em):.1f}, 吸収線: {peak_significance(s_abs):.1f}")
print("全スペクトルの輝線のピークの目立ち具合:", np.round(sig_em_all, 1))

## 4. ガウス関数の当てはめで精密に決める

おおよその $z$ が分かれば、各線の観測波長の見当がつきます。その付近 ±25 Å に、ガウス関数+定数を当てはめて、線の中心 $\lambda_{\mathrm{obs}}$ を決めます。

当てはめが怪しいもの(信号雑音比が 5 未満、中心が見当から大きくずれた、幅が不自然)は捨てます。

In [ ]:
def gauss(x, amp, mu, sigma, c0):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2) + c0


def fit_line(lam, flux, ivar, lam_guess, half_width=25.0, sign=+1):
    # 1 本の線にガウス関数を当てはめる。うまくいかなければ None
    sel = (np.abs(lam - lam_guess) < half_width) & (ivar > 0)
    if sel.sum() < 10:
        return None
    x, y, err = lam[sel], flux[sel], 1 / np.sqrt(ivar[sel])
    p0 = [sign * max(np.ptp(y), 1e-3), lam_guess, 2.5, np.median(y)]
    try:
        p, cov = curve_fit(gauss, x, y, p0=p0, sigma=err, absolute_sigma=True, maxfev=5000)
    except (RuntimeError, ValueError):
        return None
    perr = np.sqrt(np.abs(np.diag(cov)))
    ok = (np.isfinite(perr[1]) and perr[0] > 0 and np.sign(p[0]) == sign
          and abs(p[1] - lam_guess) < half_width / 2 and 0.7 < abs(p[2]) < 15)
    if not ok:
        return None
    return dict(lam_obs=p[1], lam_obs_err=perr[1], sigma=abs(p[2]), snr=abs(p[0]) / perr[0])


def measure_lines(lam, flux, ivar, z_guess, lines, sign, min_snr=5.0, max_dv=300.0):
    # 線のリストの各線について z を測る。z_guess から max_dv [km/s] 以上ずれたものは別の何かとみなして捨てる
    rows = []
    for name, (lam0, _) in lines.items():
        r = fit_line(lam, flux, ivar, lam0 * (1 + z_guess), sign=sign)
        if r is None or r["snr"] < min_snr:
            continue
        z_line = r["lam_obs"] / lam0 - 1
        if abs(z_line - z_guess) * C_KMS / (1 + z_guess) > max_dv:
            continue
        rows.append(dict(line=name, lam_rest=lam0, z=z_line, z_err=r["lam_obs_err"] / lam0, **r))
    return pd.DataFrame(rows)


z_guess = Z_GRID[np.argmax(s_em)]
lines_found = measure_lines(lam, flux, ivar, z_guess, EMISSION, sign=+1)
lines_found.round(5)

線ごとの $z$ を、誤差で重みをつけて平均します。
線どうしのばらつきが誤差より大きいときは、誤差を広げておきます(線の形が単純なガウス関数でないことなどによる)。

In [ ]:
def combine(df):
    w = 1 / df["z_err"] ** 2
    z = np.sum(w * df["z"]) / np.sum(w)
    err = 1 / np.sqrt(np.sum(w))
    if len(df) > 1:
        chi2_red = np.sum(w * (df["z"] - z) ** 2) / (len(df) - 1)
        err *= np.sqrt(max(chi2_red, 1.0))
    return z, err


z_mine, z_mine_err = combine(lines_found)
print(f"測った赤方偏移: z = {z_mine:.5f} ± {z_mine_err:.5f}  ({len(lines_found)} 本の線から)")
print(f"速度に直すと cz = {C_KMS * z_mine:,.0f} ± {C_KMS * z_mine_err:.0f} km/s")

In [ ]:
fig, axes = plt.subplots(1, len(lines_found), figsize=(3 * len(lines_found), 3), squeeze=False)
for ax, (_, r) in zip(axes[0], lines_found.iterrows()):
    sel = np.abs(lam - r["lam_obs"]) < 30
    ax.plot(lam[sel], flux[sel], "k", drawstyle="steps-mid", lw=0.8)
    ax.axvline(r["lam_rest"] * (1 + z_mine), c="tab:red", ls="--", lw=1)
    ax.set_title(f"{r['line']}\n{r['lam_rest']:.1f} → {r['lam_obs']:.1f} Å", fontsize=9)
    ax.set_xlabel("λ_obs [Å]")
plt.tight_layout()
plt.show()

## 5. 赤方偏移は「足し算」ではなく「掛け算」

観測波長を静止波長に対して並べると、原点を通る直線 $\lambda_{\mathrm{obs}} = (1+z)\,\lambda_{\mathrm{rest}}$ に乗ります。
どの線も同じ **倍率** で伸びていて、全部の線が同じ Å 数だけずれる(足し算のずれ)のではありません。
光の「波そのもの」が、空間の伸び(あるいはドップラー効果)で引き伸ばされているからです。

In [ ]:
coef = np.polyfit(lines_found["lam_rest"], lines_found["lam_obs"], 1)
shift_add = np.mean(lines_found["lam_obs"] - lines_found["lam_rest"])
print(f"直線の当てはめ: λ_obs = {coef[0]:.5f} λ_rest + {coef[1]:.2f} Å  (1 + z = {1 + z_mine:.5f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xx = np.linspace(3500, 7000, 10)
axes[0].plot(lines_found["lam_rest"], lines_found["lam_obs"], "o")
axes[0].plot(xx, (1 + z_mine) * xx, c="tab:red", label="λ_obs = (1+z) λ_rest")
axes[0].plot(xx, xx + shift_add, c="gray", ls="--", label="λ_obs = λ_rest + const")
axes[0].set_xlabel("rest wavelength [Å]")
axes[0].set_ylabel("observed wavelength [Å]")
axes[0].legend()
axes[1].errorbar(lines_found["lam_rest"], lines_found["lam_obs"] - lines_found["lam_rest"],
                 yerr=lines_found["lam_obs_err"], fmt="o")
axes[1].plot(xx, z_mine * xx, c="tab:red", label="Δλ = z λ_rest")
axes[1].axhline(shift_add, c="gray", ls="--", label="Δλ = const")
axes[1].set_xlabel("rest wavelength [Å]")
axes[1].set_ylabel("Δλ = λ_obs − λ_rest [Å]")
axes[1].legend()
plt.tight_layout()
plt.show()

右の図で、ずれ $\Delta\lambda$ は静止波長に比例して大きくなっていて、一定(灰色の破線)ではありません。

## 6. 全部の銀河で測って、SDSS の値と比べる

2〜4 章の手順を関数にまとめて、全部のスペクトルに使います。

- 輝線の相互相関のピークがはっきりしていれば(目立ち具合 8 以上)輝線を使う。輝線は細くて位置が精度よく決まるから。そうでなければ、ピークがよりはっきりしているほうを使う
- 採用した線のリストでガウス関数を当てはめ、2 本以上の線が見つかれば、その平均を自分の $z$ にする
- 線が 1 本以下なら、相互相関のピークの位置そのもの(精度は格子の幅 0.0002 程度)を使う

In [ ]:
def measure_redshift(lam, flux, ivar, s_em, s_abs):
    sig_em, sig_abs = peak_significance(s_em), peak_significance(s_abs)
    if sig_em >= EMISSION_THRESHOLD or sig_em >= sig_abs:
        method, lines, sign, score = "emission", EMISSION, +1, s_em
    else:
        method, lines, sign, score = "absorption", ABSORPTION, -1, s_abs
    z_guess = Z_GRID[np.argmax(score)]
    found = measure_lines(lam, flux, ivar, z_guess, lines, sign=sign, min_snr=5.0 if sign > 0 else 3.0)
    if len(found) >= 2:
        z, err = combine(found)
    else:
        z, err = z_guess, Z_GRID[1] - Z_GRID[0]
    return dict(z_mine=z, z_mine_err=err, method=method, n_lines=len(found), significance=max(sig_em, sig_abs))


results = pd.DataFrame([measure_redshift(*s, *sc) for s, sc in zip(spectra, scores)])
compare = pd.concat([targets[["plate", "mjd", "fiberid", "z", "zerr", "subclass"]], results], axis=1)
compare["dv_kms"] = C_KMS * (compare["z_mine"] - compare["z"]) / (1 + compare["z"])
compare.round({"z": 5, "zerr": 5, "z_mine": 5, "z_mine_err": 5, "significance": 1, "dv_kms": 0})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method, marker in [("emission", "o"), ("absorption", "s")]:
    m = compare["method"] == method
    axes[0].errorbar(compare.loc[m, "z"], compare.loc[m, "z_mine"], yerr=compare.loc[m, "z_mine_err"], fmt=marker, label=method)
    axes[1].errorbar(compare.loc[m, "z"], compare.loc[m, "dv_kms"], yerr=C_KMS * compare.loc[m, "z_mine_err"], fmt=marker, label=method)
zz = np.linspace(0, compare["z"].max() * 1.05, 10)
axes[0].plot(zz, zz, c="gray", lw=1)
axes[0].set_xlabel("z (SDSS pipeline)")
axes[0].set_ylabel("z (this notebook)")
axes[1].axhline(0, c="gray", lw=1)
axes[1].set_xlabel("z (SDSS pipeline)")
axes[1].set_ylabel("c Δz / (1+z) [km/s]")
axes[1].set_ylim(-300, 300)
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

good = compare["n_lines"] >= 2
print(f"線が 2 本以上見つかった銀河: {good.sum()} / {len(compare)}")
print(f"SDSS との差: 中央値 {compare.loc[good, 'dv_kms'].median():+.0f} km/s, "
      f"ばらつき(標準偏差) {compare.loc[good, 'dv_kms'].std():.0f} km/s")
bad = np.abs(compare["dv_kms"]) > 500
if bad.any():
    print("大きく外れた銀河:")
    print(compare.loc[bad, ["plate", "mjd", "fiberid", "z", "z_mine", "method", "n_lines"]])

輝線のはっきりした銀河では、SDSS の値と数十 km/s 以内($\Delta z \sim 10^{-4}$)で一致するはずです。
吸収線だけの銀河は線が幅広く浅いので、誤差は大きめ(100 km/s 程度)になります。

SDSS のパイプラインは、線のリストではなく、たくさんの銀河から作った「主成分」の型紙をスペクトル全体に当てはめて $z$ を決めています。
考え方(型紙をずらして一番合うところを探す)は、このノートブックの相互相関と同じです。

> 大きく外れた銀河がある場合は、相互相関で別の線どうしを取り違えている(例えば [O III] 5008 と Hα を取り違える)ことが多いです。
> `find_z` の $S(z)$ を描いて、ピークが複数ないか見てみてください。

## まとめ

- 線のリストを $z$ だけ伸ばした型紙との相互相関で、おおよその $z$ が見つかる
- 1 本ずつガウス関数を当てはめると、輝線銀河では $\Delta z \sim 10^{-5}$〜$10^{-4}$($\sim$10 km/s)の精度で $z$ が決まり、SDSS の値と一致する
- どの線も同じ倍率 $1+z$ で伸びている

## 発展メモ

- クエーサー(`class = 'QSO'`)を選び、Lyα(1215.7 Å)や C IV(1549 Å)などの紫外の線のリストを足して、$z = 2$〜3 の天体の $z$ を測る
- 輝線の幅(ガウス関数の σ)から、ガスの速度の広がりを km/s で出す(SDSS の分光器の分解能 約 70 km/s を差し引く)
- 輝線の強さの比 [N II]/Hα と [O III]/Hβ を並べて(BPT 図)、星形成銀河と活動銀河核を分ける
- 自分で測った $z$ と、銀河の明るさからおおよその距離を出して、ハッブル図を描いてみる

## 参考・クレジット

- Abdurro'uf et al. (2022), *The Seventeenth Data Release of the Sloan Digital Sky Surveys*, ApJS 259, 35
- Bolton et al. (2012), *Spectral Classification and Redshift Measurement for the SDSS-III Baryon Oscillation Spectroscopic Survey*, AJ 144, 144(SDSS のパイプラインの $z$ の決め方)
- 線の真空中の波長: SDSS の線のリスト(https://classic.sdss.org/dr7/products/spectra/vacwavelength.php)
- Funding for the Sloan Digital Sky Survey IV has been provided by the Alfred P. Sloan Foundation, the U.S. Department of Energy Office of Science, and the Participating Institutions. SDSS acknowledges support and resources from the Center for High-Performance Computing at the University of Utah. The SDSS web site is www.sdss4.org.